<a href="https://colab.research.google.com/github/Sr1n1vas0504/SMART-SUSPENSION-SYSTEM/blob/main/Final_Damper_Settings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install ipywidgets joblib
!pip install ipywidgets

In [6]:
# ==========================================
# SMART SUSPENSION ARX PREDICTOR
# Opens output in NEW TAB (Colab Safe)
# ==========================================

import numpy as np
import pandas as pd
import joblib
import ipywidgets as widgets
import re
import base64
from IPython.display import display, clear_output, HTML

# -------------------------------
# LOAD MODEL + SCALERS
# -------------------------------
model = joblib.load("arx_model.pkl")
scaler_X = joblib.load("scaler_X.pkl")
scaler_y = joblib.load("scaler_y.pkl")

# Auto-detect correct feature count and order
input_fields = list(scaler_X.feature_names_in_)
required_features = scaler_X.n_features_in_

# -------------------------------
# UI
# -------------------------------
title = HTML("<h2>🚗 Smart Suspension ARX Predictor</h2>")
instruction = HTML(f"<b>Enter exactly {required_features} values separated by comma / tab / space</b>")

input_box = widgets.Textarea(
    placeholder="Paste 34 values here...",
    layout=widgets.Layout(width='950px', height='120px')
)

predict_button = widgets.Button(
    description="Open Prediction Report",
    button_style='success'
)

output = widgets.Output()

# -------------------------------
# PREDICTION FUNCTION
# -------------------------------
def predict(b):
    with output:
        clear_output()

        try:
            # Split by comma / tab / space
            raw_values = re.split(r'[,\t\s]+', input_box.value.strip())
            values = [float(x) for x in raw_values if x != ""]

            if len(values) != required_features:
                print(f"❌ Expected {required_features} values but got {len(values)}")
                return

            # Create dataframe in correct feature order
            input_df = pd.DataFrame([values], columns=input_fields)

            # Scale
            input_scaled = scaler_X.transform(input_df)

            # Predict
            pred_scaled = model.predict(input_scaled)
            prediction = scaler_y.inverse_transform(pred_scaled)[0]

            # -------------------------------
            # HTML REPORT
            # -------------------------------
            html_content = f"""
            <html>
            <head>
            <title>Smart Suspension Result</title>
            <style>
                body {{
                    font-family: Arial;
                    background-color: #f4f6f9;
                    padding: 40px;
                }}
                .container {{
                    background: white;
                    padding: 30px;
                    border-radius: 12px;
                    box-shadow: 0 10px 25px rgba(0,0,0,0.1);
                }}
                h2 {{
                    color: #1e3a8a;
                }}
                .result {{
                    background: #eef2ff;
                    padding: 20px;
                    border-radius: 10px;
                    margin-bottom: 25px;
                }}
                table {{
                    border-collapse: collapse;
                    width: 100%;
                }}
                th, td {{
                    border: 1px solid #ddd;
                    padding: 8px;
                    font-size: 13px;
                }}
                th {{
                    background-color: #f2f2f2;
                }}
            </style>
            </head>
            <body>
            <div class="container">
                <h2>Smart Suspension Prediction Report</h2>

                <div class="result">
                    <h3>Prediction Results</h3>
                    <p><b>Optimal Damper Coefficient:</b> {prediction[0]:.4f}</p>
                    <p><b>Optimal Damper Force (N):</b> {prediction[1]:.4f}</p>
                    <p><b>Optimal Damper Displacement (m):</b> {prediction[2]:.6f}</p>
                </div>

                <h3>Input Values Used</h3>
                <table>
                    <tr>
                        <th>Feature</th>
                        <th>Value</th>
                    </tr>
            """

            for name, value in zip(input_fields, values):
                html_content += f"""
                    <tr>
                        <td>{name}</td>
                        <td>{value}</td>
                    </tr>
                """

            html_content += """
                </table>
            </div>
            </body>
            </html>
            """

            # Encode HTML for new tab opening
            html_bytes = html_content.encode("utf-8")
            encoded = base64.b64encode(html_bytes).decode()
            data_url = f"data:text/html;base64,{encoded}"

            print("✅ Click below to open result in new tab:\n")
            display(HTML(f'<a href="{data_url}" target="_blank" style="font-size:18px;">🔗 Open Smart Suspension Report</a>'))

        except Exception as e:
            print("❌ ERROR:")
            print(e)

predict_button.on_click(predict)

# -------------------------------
# DISPLAY UI
# -------------------------------
display(title)
display(instruction)
display(input_box)
display(predict_button)
display(output)

Textarea(value='', layout=Layout(height='120px', width='950px'), placeholder='Paste 34 values here...')

Button(button_style='success', description='Open Prediction Report', style=ButtonStyle())

Output()